# OBGPU HFO Regime Optimizer

This notebook targets the provisional EPLI TC-only network and searches for parameter regimes that produce a clean LFP band near 180 +/- 20 Hz under ketamine-like NMDA block, while staying weaker in the paired control condition.

Important execution rule: this notebook is designed to reuse an existing manual Phoenix allocation by setting `slurm_allocation_job_id`. It should not request a fresh long-lived allocation of its own.

In [ ]:
from pathlib import Path

import pandas as pd

import obgpu_experiment_helpers as hlp
from olfactorybulb.hfo_optimizer import (
    build_manual_allocation_remote_config,
    default_campaign_run_config,
    default_hfo_search_space,
    ensure_campaign_dir,
    infer_remote_template_from_recent_runs,
    initialize_campaign,
    load_candidate_archive_rows,
    maybe_dataframe,
    paramiko_auth_probe,
    propose_elite_batch,
    propose_lhs_batch,
    run_hfo_batch,
    score_hfo_batch,
    search_space_rows,
    top_candidate_rows,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [ ]:
REMOTE_TEMPLATE = infer_remote_template_from_recent_runs()

MANUAL_ALLOCATION_JOB_ID = "PASTE_EXISTING_7DAY_ALLOCATION_ID_HERE"
TOTAL_TASKS = 120
ITEM_NRANKS = 15

REMOTE_CONFIG = build_manual_allocation_remote_config(
    slurm_allocation_job_id=MANUAL_ALLOCATION_JOB_ID,
    base_template=REMOTE_TEMPLATE,
    total_tasks=TOTAL_TASKS,
)

pd.Series({
    "remote_host": REMOTE_CONFIG["remote_host"],
    "remote_repo_root": REMOTE_CONFIG["remote_repo_root"],
    "remote_results_root": REMOTE_CONFIG["remote_results_root"],
    "remote_mpi_exec": REMOTE_CONFIG["remote_mpi_exec"],
    "slurm_allocation_job_id": REMOTE_CONFIG["slurm_allocation_job_id"],
    "optimizer_total_tasks": REMOTE_CONFIG["optimizer_total_tasks"],
    "item_nranks": ITEM_NRANKS,
    "batch_parallelism": TOTAL_TASKS // ITEM_NRANKS,
})

In [ ]:
if "PASTE_EXISTING" in MANUAL_ALLOCATION_JOB_ID:
    raise ValueError("Set MANUAL_ALLOCATION_JOB_ID to the already-running 7-day Phoenix allocation before probing auth.")

AUTH_PROBE = paramiko_auth_probe(REMOTE_CONFIG)
AUTH_PROBE

## Campaign setup

The optimizer uses a batch-first search:
- Latin-hypercube seed batch for global coverage
- paired `control` and `ketamine` runs for every candidate
- elite-centered refinement batches after scoring

Time constants stay fixed. The search varies conductance-like and coupling-like knobs only.

In [ ]:
CAMPAIGN_NAME = f"hfo_epli_tc_only_{hlp.make_timestamp()}"
CAMPAIGN_DIR = ensure_campaign_dir(CAMPAIGN_NAME)
SEARCH_SPACE = default_hfo_search_space()

BASE_CONFIG = default_campaign_run_config(
    REMOTE_CONFIG,
    paramset="GammaSignature_EPLI_Provisional_TCOnly",
    nranks=ITEM_NRANKS,
    total_tasks=TOTAL_TASKS,
    tstop_ms=9000.0,
    cell_permute=0,
)

initialize_campaign(
    CAMPAIGN_DIR,
    base_config=BASE_CONFIG,
    search_space=SEARCH_SPACE,
    notes="Manual-allocation Phoenix HFO search for provisional EPLI TC-only regime.",
)

display(pd.Series({
    "campaign_dir": str(CAMPAIGN_DIR),
    "paramset": BASE_CONFIG["paramset"],
    "nranks": BASE_CONFIG["nranks"],
    "sweep_parallelism": BASE_CONFIG["sweep_parallelism"],
    "tstop_ms": BASE_CONFIG["tstop_ms"],
    "use_corenrn": BASE_CONFIG["use_corenrn"],
    "cell_permute": BASE_CONFIG["cell_permute"],
})))
pd.DataFrame(search_space_rows(SEARCH_SPACE))

In [ ]:
INITIAL_CANDIDATES = 24
INITIAL_SEED = 20260527

batch0 = propose_lhs_batch(
    CAMPAIGN_DIR,
    search_space=SEARCH_SPACE,
    n_candidates=INITIAL_CANDIDATES,
    seed=INITIAL_SEED,
    stage="screen",
)

pd.DataFrame(batch0["candidates"]).head()

In [ ]:
sweep0 = run_hfo_batch(
    CAMPAIGN_DIR,
    base_config=BASE_CONFIG,
    batch_plan=batch0,
)

sweep0["sweep_dir"]

In [ ]:
scored0 = score_hfo_batch(
    CAMPAIGN_DIR,
    batch_plan=batch0,
    sweep=sweep0,
    signal="lfp",
    dt_ms=0.1,
    target_hz=180.0,
    target_half_width_hz=20.0,
)

leaders0 = sorted(scored0["candidate_rows"], key=lambda row: row["pair_score"], reverse=True)
maybe_dataframe(leaders0[:12])

In [ ]:
batch1 = propose_elite_batch(
    CAMPAIGN_DIR,
    search_space=SEARCH_SPACE,
    n_candidates=24,
    seed=20260528,
    elite_frac=0.25,
    explore_frac=0.25,
    stage="refine",
)

pd.DataFrame(batch1["candidates"]).head()

In [ ]:
maybe_dataframe(top_candidate_rows(CAMPAIGN_DIR, limit=20))